# Agentic RAG App — Phase 3
**Author:** Shruti  
**Description:** A hybrid AI agent combining RAG (Retrieval-Augmented Generation) with tool-calling agents. The agent can search documents, summarize text, perform calculations, and answer questions with memory across turns.

**Stack:** Python, LangChain, LangGraph, ChromaDB, HuggingFace Embeddings, Groq API (Qwen3-32B)

---

## Setup
Install libraries and configure dependencies.

In [ ]:
!pip install langchain langchain-community langchain-text-splitters langchain-groq langchain-huggingface chromadb sentence-transformers groq langgraph -q

from langchain_groq import ChatGroq
from langchain.agents import create_react_agent
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
import json

# Replace with your Groq API key from console.groq.com
GROQ_API_KEY = "your-key-here"

llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model="qwen/qwen3-32b"
)

print("Setup complete!")

---
## Day 11 — LangChain Agents & Tools

**Skills practiced:** Creating custom tools with `@tool`, building an agent that chooses between tools, chaining multiple tools in sequence.

### Exercises 1 & 2 — Create tools and first agent
Define custom tools and create an agent that decides which tool to use.

In [ ]:
@tool
def calculate(expression: str) -> str:
    """Evaluate a math expression like 999*111 or 5**2."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

@tool
def get_ai_info(topic: str) -> str:
    """Get information about an AI topic: rag, embeddings, machine learning, or llm."""
    info = {
        "rag": "Retrieval-Augmented Generation combines retrieval with text generation.",
        "embeddings": "Embeddings are numerical representations of text that capture meaning.",
        "machine learning": "Machine Learning enables computers to learn from data.",
        "llm": "Large Language Models are trained on vast text data to understand language."
    }
    return info.get(topic.lower().strip(), f"No info found for '{topic}'.")

@tool
def word_count(expression: str) -> str:
    """Count the number of words in a string."""
    return str(len(expression.split()))

agent = create_react_agent(llm, tools=[calculate, get_ai_info, word_count])
print("Agent created with 3 tools!")

### Exercises 3, 4 & 5 — Test agent with multiple questions and tool chaining
Test the agent choosing different tools and chaining them in sequence.

In [ ]:
# Test 1: Math
response = agent.invoke({"messages": [{"role": "user", "content": "What is 999 multiplied by 111?"}]})
print("Math Result:", response["messages"][-1].content)

# Test 2: AI Info
response = agent.invoke({"messages": [{"role": "user", "content": "Tell me about RAG."}]})
print("\nAI Info Result:", response["messages"][-1].content)

# Test 3: Word count
response = agent.invoke({"messages": [{"role": "user", "content": "How many words are in: The quick brown fox jumps over the lazy dog"}]})
print("\nWord Count Result:", response["messages"][-1].content)

# Test 4: Multi-tool chain
response = agent.invoke({"messages": [{"role": "user", "content": "What is the word count of the RAG description multiplied by 2?"}]})
print("\nChained Result:", response["messages"][-1].content)

---
## Day 12 — Hybrid RAG Agent

**Skills practiced:** Wrapping a RAG pipeline as an agent tool, combining document search with summarization, building a hybrid RAG + Agent system.

### Exercise 1 — Build the RAG knowledge base
Create a knowledge base and retriever to be used by the agent.

In [ ]:
text = """
Machine Learning is a subset of AI that enables computers to learn from data without being explicitly programmed. It uses algorithms to find patterns and make predictions. Common types include supervised, unsupervised, and reinforcement learning.

Retrieval-Augmented Generation (RAG) combines information retrieval with text generation. It retrieves relevant documents from a knowledge base and uses them as context for the LLM. RAG reduces hallucinations by grounding responses in real data.

Embeddings are numerical representations of text that capture semantic meaning. Similar texts produce similar embeddings. They are used in search, recommendation systems, and RAG pipelines.

Large Language Models (LLMs) are trained on vast amounts of text data to understand and generate human language. Examples include GPT-4, Claude, and LLaMA. They power applications like summarization, translation, and question answering.

Vector databases store embeddings and enable fast similarity search. Examples include ChromaDB, Pinecone, and Weaviate. They are essential components of RAG systems.
"""

with open("tech_knowledge.txt", "w") as f:
    f.write(text)

with open("tech_knowledge.txt", "r") as f:
    text = f.read()

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.split_text(text)

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_texts(
    texts=chunks,
    embedding=embeddings,
    collection_name="day12_knowledge"
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print(f"Stored {vectorstore._collection.count()} chunks!")

# Test retriever
docs = retriever.invoke("what is RAG?")
for doc in docs:
    print(doc.page_content[:100], "...")

### Exercise 2 — Write the `search_documents` tool
Wrap the retriever inside a tool so the agent can search the knowledge base.

In [ ]:
@tool
def search_documents(query: str) -> str:
    """Search the knowledge base for information about AI topics like RAG, embeddings, LLMs, and machine learning."""
    docs = retriever.invoke(query)
    return "\n".join([doc.page_content for doc in docs])

# Test the tool directly
print(search_documents.invoke("what are embeddings?"))

### Exercise 3 — Write the `summarize_text` tool
Create a tool that summarizes long text using the LLM.

In [ ]:
@tool
def summarize_text(text: str) -> str:
    """Summarize a long text into 2 concise sentences."""
    response = llm.invoke([
        SystemMessage(content="You are a helpful assistant that summarizes text concisely."),
        HumanMessage(content=f"Summarize this in 2 sentences: {text}")
    ])
    return response.content

# Test the tool directly
print(summarize_text.invoke("Machine learning is a subset of AI that enables computers to learn from data. It uses algorithms to find patterns and make predictions based on those patterns."))

### Exercises 4 & 5 — Wire everything into one hybrid agent
Create an agent with all tools and test multi-tool chaining.

In [ ]:
hybrid_agent = create_react_agent(
    llm,
    tools=[calculate, search_documents, summarize_text]
)

# Test 1: Search and summarize
response = hybrid_agent.invoke({"messages": [{"role": "user", "content": "Search for information about embeddings and summarize it"}]})
print("Search + Summarize:", response["messages"][-1].content)

# Test 2: RAG question
response = hybrid_agent.invoke({"messages": [{"role": "user", "content": "What is RAG? Give me a short summary"}]})
print("\nRAG Summary:", response["messages"][-1].content)

# Test 3: Math still works
response = hybrid_agent.invoke({"messages": [{"role": "user", "content": "What is 256 times 4?"}]})
print("\nMath Result:", response["messages"][-1].content)

# Test 4: Multi-tool chain
response = hybrid_agent.invoke({"messages": [{"role": "user", "content": "Search for information about large language models and then summarize what you find"}]})
print("\nChained Result:", response["messages"][-1].content)

---
## Day 13 — Final Project: Interactive Agentic RAG App

**Skills practiced:** Combining all tools into one agent, adding memory across conversation turns, building an interactive agent loop.

**This is the Phase 3 final project** — a complete agentic app with RAG, summarization, math, and memory.

### Exercises 1 & 2 — Add `answer_from_document` tool
Wrap the full RAG pipeline as a single tool that retrieves and answers in one step.

In [ ]:
@tool
def answer_from_document(question: str) -> str:
    """Answer a question using the knowledge base documents. Use this for factual questions about AI topics."""
    docs = retriever.invoke(question)
    context = "\n".join([doc.page_content for doc in docs])
    response = llm.invoke([
        SystemMessage(content=f"You are a helpful assistant. Use this context to answer: {context}"),
        HumanMessage(content=question)
    ])
    return response.content

# Test the tool
print(answer_from_document.invoke("What is the difference between RAG and fine-tuning?"))

### Exercise 3 — Build the final agent with all 4 tools

In [ ]:
final_agent = create_react_agent(
    llm,
    tools=[calculate, search_documents, summarize_text, answer_from_document]
)

# Test all tools
response = final_agent.invoke({"messages": [{"role": "user", "content": "What does the document say about deep learning?"}]})
print("Deep Learning:", response["messages"][-1].content)

response = final_agent.invoke({"messages": [{"role": "user", "content": "Search for embeddings and give me a grounded answer"}]})
print("\nEmbeddings:", response["messages"][-1].content)

response = final_agent.invoke({"messages": [{"role": "user", "content": "What is 512 divided by 16?"}]})
print("\nMath:", response["messages"][-1].content)

### Exercise 4 — Add memory across conversation turns
Pass the full message history so the agent remembers previous answers.

In [ ]:
messages = []

# Turn 1
question = "What is RAG?"
messages.append({"role": "user", "content": question})
response = final_agent.invoke({"messages": messages})
answer = response["messages"][-1].content
messages.append({"role": "assistant", "content": answer})
print("Answer 1:", answer)

# Turn 2 — follow-up referring to previous answer
followup = "Can you summarize what you just told me in one sentence?"
messages.append({"role": "user", "content": followup})
response = final_agent.invoke({"messages": messages})
answer2 = response["messages"][-1].content
messages.append({"role": "assistant", "content": answer2})
print("\nAnswer 2:", answer2)

### Exercise 5 — Interactive agent loop
Final project: interactive agent with memory, RAG, summarization, and math tools.

In [ ]:
messages = []

print("AI Agent ready! Ask anything about AI topics, or ask for calculations.")
print("Type 'quit' to exit.\n")

while True:
    question = input("You: ")
    if question.lower() == "quit":
        print("Goodbye!")
        break

    messages.append({"role": "user", "content": question})
    response = final_agent.invoke({"messages": messages})
    answer = response["messages"][-1].content
    messages.append({"role": "assistant", "content": answer})

    print(f"\nAgent: {answer}")
    print("\n" + "-" * 50 + "\n")

---
## Phase 3 Summary

**What this project builds:**
- Custom LangChain tools using the `@tool` decorator
- An agent that autonomously decides which tools to use
- A hybrid RAG + Agent system
- Multi-turn memory across conversation
- Interactive CLI agent app

**Key concepts learned:**
- Tool creation and docstring engineering
- Agent reasoning and tool selection
- Chaining multiple tools in sequence
- Combining RAG retrieval with agent decision-making
- Conversation memory management

**Tools built:**
- `calculate` — math evaluation
- `get_ai_info` — hardcoded knowledge lookup
- `word_count` — string analysis
- `search_documents` — RAG retrieval tool
- `summarize_text` — LLM summarization tool
- `answer_from_document` — full RAG pipeline as tool